In [0]:
#display(dbutils.fs.ls("/Volumes/marathos/default/raw/"))

#%restart_python

In [0]:
#place it in a _setup notebook inside the explorations folder

#need to run this start of each cell one time for the notebook to work, dont like having filepaths, so I did this

#import sys

#PROJECT_ROOT = "/Workspace/Users/<email>/<projectname(root)/<pyproject location> if pyproject is in root this last one isnt needed"

#if PROJECT_ROOT not in sys.path:
    #sys.path.insert(0, PROJECT_ROOT)

In [0]:
%run ../explorations/_setup

In [0]:
%skip
from importlib import reload
import transformations.bronze.ingest_bronze as bronze_module
reload(bronze_module)

In [0]:
from utils.helpers import get_table
from pyspark.sql import functions as F
from transformations.bronze.ingest_bronze import ingest_bronze

ingest_bronze(spark)

In [0]:
df_bronze = get_table("marathos.bronze.raw_results")

# Check for leading zeros or whitespace in athlete_id_raw
df_bronze.select("athlete_id_raw") \
  .filter(F.col("athlete_id_raw").startswith(" ") | 
          F.col("athlete_id_raw").startswith("0") & (F.length("athlete_id_raw") > 1)) \
  .limit(10) \
  .display()

In [0]:
df_bronze = get_table("marathos.bronze.raw_results")
# Check the max and total unique values
df_bronze.agg(
    F.countDistinct("athlete_id_raw").alias("unique_athlete_ids"),
    F.max("athlete_id_raw").alias("max_athlete_id"),
    F.count("*").alias("total_rows")
).display()

In [0]:
# Also check the actual numeric max by casting first
df_bronze.agg(
    F.max(F.col("athlete_id_raw").cast("long")).alias("max_id_numeric"),
    F.min(F.col("athlete_id_raw").cast("long")).alias("min_id_numeric"),
    F.countDistinct(F.col("athlete_id_raw").cast("long")).alias("unique_ids_numeric"),
    F.countDistinct("athlete_id_raw").alias("unique_ids_string"),
    F.count("*").alias("total_rows")
).display()

In [0]:
df_bronze = get_table("marathos.bronze.raw_results")

df_bronze.filter(
    (F.col("athlete_id_raw") == "4033") &
    (F.col("event_name") == "Tagaytay to Kawit 50K Ultramarathon (PHI)") &
    (F.col("event_dates") == "24.-25.08.2013")
).select(
    "athlete_id_raw",
    "athlete_performance_raw",
    "athlete_country",
    "athlete_year_of_birth",
    "athlete_gender",
    "athlete_club"
).display()

In [0]:
# Also check if athlete_id_raw resets to 0 at each new event
df_bronze.groupBy("event_name", "event_dates") \
  .agg(
    F.min("athlete_id_raw").alias("min_id"),
    F.max("athlete_id_raw").alias("max_id"),
    F.count("*").alias("finishers")
  ) \
  .orderBy("min_id") \
  .limit(10) \
  .display()

In [0]:
df_bronze = get_table("marathos.bronze.raw_results")

# Test 1: Is athlete_id_raw truly sequential with no gaps?
# If max = count - 1 then it's a clean 0-based sequence
total_rows = df_bronze.count()
max_id = df_bronze.agg(F.max("athlete_id_raw")).collect()[0][0]
min_id = df_bronze.agg(F.min("athlete_id_raw")).collect()[0][0]

print(f"Total rows:  {total_rows}")
print(f"Min ID:      {min_id}")
print(f"Max ID:      {max_id}")
print(f"Expected max if sequential: {total_rows - 1}")
print(f"Truly sequential: {int(max_id) == total_rows - 1}")

In [0]:
# Test 2: Are there any gaps in the sequence?
# If unique count == max + 1 then no gaps
unique_ids = df_bronze.select("athlete_id_raw").distinct().count()
print(f"Unique IDs:  {unique_ids}")
print(f"No gaps: {unique_ids == int(max_id) + 1}")

In [0]:
df_bronze = get_table("marathos.bronze.raw_results")
df_bronze.printSchema()

In [0]:
# Test 3: Does the same athlete_id_raw always have the same country and gender?
# If truly unique per person, these should never differ
df_bronze.groupBy("athlete_id_raw") \
  .agg(
    F.countDistinct("athlete_country").alias("unique_countries"),
    F.countDistinct("athlete_gender").alias("unique_genders"),
    F.countDistinct("athlete_year_of_birth").alias("unique_birth_years")
  ) \
  .filter(
    (F.col("unique_countries") > 1) |
    (F.col("unique_genders") > 1) |
    (F.col("unique_birth_years") > 1)
  ) \
  .count()

In [0]:
df_bronze = get_table("marathos.bronze.raw_results")

df_bronze.groupBy("athlete_id_raw", "event_name", "event_dates") \
  .count() \
  .filter(F.col("count") > 1) \
  .orderBy(F.desc("count")) \
  .limit(10) \
  .display()

In [0]:
df_bronze = get_table("marathos.bronze.raw_results")

# Check if athlete_id_raw is unique across the entire dataset
df_bronze.groupBy("athlete_id_raw") \
  .count() \
  .orderBy(F.desc("count")) \
  .limit(10) \
  .display()

In [0]:
df = get_table("marathos.bronze.raw_results")

In [0]:
print(f"Number of rows:    {df.count()}")
print(f"Number of columns: {len(df.columns)}")

In [0]:
#bronze layer
df.printSchema()

In [0]:
df.limit(5).display()

In [0]:
df.describe().display()

In [0]:
null_counts = df.select([
    F.count(
        F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), c)
    ).alias(c)
    for c in df.columns
])

null_counts.display()

In [0]:
unique_events = df.select("event_name").distinct().count()
print(f"Number of unique events: {unique_events}")

In [0]:
df.select("event_name").distinct().limit(20).display()

In [0]:
df.groupBy("event_distance_raw") \
  .count() \
  .orderBy(F.desc("count")) \
  .display()

In [0]:
df.select("athlete_performance_raw") \
  .distinct() \
  .limit(20) \
  .display()

In [0]:
df.withColumn(
    "birth_year", F.col("athlete_year_of_birth").cast("double").cast("int")
).withColumn(
    "event_year", F.col("year_of_event").cast("int")
).withColumn(
    "age", F.col("event_year") - F.col("birth_year")
).groupBy("age") \
 .count() \
 .orderBy("age") \
 .filter(F.col("age").isNotNull()) \
 .display()

In [0]:
df.groupBy("athlete_country") \
  .count() \
  .orderBy(F.desc("count")) \
  .limit(20) \
  .display()

In [0]:
df.groupBy("athlete_gender") \
  .count() \
  .orderBy(F.desc("count")) \
  .display()

In [0]:
df.groupBy("year_of_event") \
  .count() \
  .orderBy("year_of_event") \
  .display()

some checks for bronze layer above.

In [0]:
df_bronze.groupBy("event_distance_raw").count().orderBy(F.desc("count")).limit(20).display()

In [0]:
df_bronze.filter(
    F.col("event_distance_raw").like("%h%")
).select("event_distance_raw", "athlete_performance_raw") \
 .limit(10) \
 .display()

In [0]:
df_bronze.filter(
    F.col("event_distance_raw") == "12h"
).select("event_distance_raw", "athlete_performance_raw", "athlete_id_raw") \
 .limit(10) \
 .display()

In [0]:
df_bronze.groupBy("athlete_id_raw") \
  .agg(
    F.countDistinct("athlete_performance_raw").alias("unique_performances"),
    F.countDistinct("athlete_country").alias("unique_countries"),
    F.count("*").alias("total_rows")
  ) \
  .filter(F.col("unique_performances") > 1) \
  .orderBy(F.desc("total_rows")) \
  .limit(10) \
  .display()

In [0]:
df_bronze.groupBy(
    "athlete_country",
    "athlete_year_of_birth", 
    "athlete_gender",
    "athlete_club"
) \
.count() \
.orderBy(F.desc("count")) \
.limit(10) \
.display()

In [0]:
#check if event_name + year_of_event is unique
df_bronze.groupBy("event_name", "year_of_event") \
  .count() \
  .orderBy(F.desc("count")) \
  .limit(10) \
  .display()

In [0]:
#need to check if there are multiple event_dates for the same event_name + year_of_event
df_bronze.groupBy("event_name", "year_of_event") \
  .agg(F.countDistinct("event_dates").alias("unique_dates")) \
  .filter(F.col("unique_dates") > 1) \
  .orderBy(F.desc("unique_dates")) \
  .limit(10) \
  .display()